# Architecture Overview: AgentCore Registry & Gateway

This notebook introduces the three pillars of the AgentCore platform and the three personas that interact with them.

> This is a reference notebook — no AWS resources are created or modified.

## Three Pillars

| Pillar | Service | Purpose |
|---|---|---|
| **Registry** | Amazon Bedrock AgentCore Registry | Discovery catalog — semantic search for tools and agents |
| **Gateway** | Amazon Bedrock AgentCore Gateway | Governed MCP endpoint — JWT auth, interceptors, guardrails |
| **Identity** | Amazon Cognito + WorkloadIdentity | Human and machine authentication |

## Three Personas

| Persona | IAM Role | Key Permissions |
|---|---|---|
| **Admin** | `workshop-ac-registry-admin-<region>` | Full CRUD + approve/reject records |
| **Publisher** | `workshop-ac-registry-publisher-<region>` | Create/update/submit records — cannot approve |
| **Consumer** | `workshop-ac-registry-consumer-<region>` | Read-only + semantic search |

## Architecture Diagram

```
┌─────────────────────────────────────────────────────┐
│                 AgentCore Platform                    │
│                                                       │
│  ┌──────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │ Registry  │  │   Gateway     │  │  Identity    │   │
│  │(Discovery)│  │ (Governance)  │  │  (Auth)      │   │
│  │           │  │               │  │              │   │
│  │ MCP tools │  │ JWT auth      │  │ Cognito Pool │   │
│  │ A2A agents│  │ Lambda targets│  │ M2M client   │   │
│  │ Search    │  │ Interceptors  │  │ WorkloadID   │   │
│  │ Approval  │  │ Guardrails    │  │ Groups       │   │
│  └──────────┘  └──────┬───────┘  └──────┬───────┘   │
│                        │                  │           │
│              ┌─────────┴──────────────────┘           │
│              │   Token validation                     │
│              ▼                                        │
│    ┌─────────────────┐                                │
│    │  Lambda Tools    │                                │
│    │ flights · hotels │                                │
│    │ search-kb        │                                │
│    └─────────────────┘                                │
└─────────────────────────────────────────────────────┘
```

## Tool Lifecycle

1. **Deploy** — CFN stack creates Lambda functions, Gateway, and targets
2. **Register** — Publisher creates MCP records in the Registry
3. **Approve** — Admin reviews and approves records
4. **Discover** — Consumer searches the Registry catalog
5. **Invoke** — Consumer calls tools via the Gateway with JWT auth

> **CFN vs API boundary:** Gateway + targets are deployed via CloudFormation. Registry + records are created via API (no CFN resource type exists).

## What the CFN Stack Deploys

| Category | Count | Examples |
|---|---|---|
| Cognito Identity | 12 | User Pool, Domain, Resource Server, 2 Clients, 3 Groups, 2 Secrets, Init Lambda |
| Persona IAM Roles | 3 | Admin, Publisher, Consumer |
| Lambda MCP Tools | 5 | flights, hotels, search-kb, product-info, order-processing |
| Gateway + Targets | 4 | 1 Gateway, 3 Lambda targets |
| Interceptors | 2 | Request (audit/ACL), Response (guardrails/sanitize) |
| WorkloadIdentity | 1 | Agent M2M identity |

**Next:** Notebook 02 deploys the stack.